# Results

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, "..")

import torch
import numpy as np
import trimesh

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Kaggle:
# ARMADILLO_PATH  = Path("/kaggle/input/your-dataset/modele/armadillo_small.obj")
# TEAPOT_PATH     = Path("/kaggle/input/your-dataset/modele/teapot.obj")
# ASIAN_DRAG_PATH = Path("/kaggle/input/your-dataset/modele/asian_dragon_really_small.obj")
# BUNNY_PATH      = Path("/kaggle/input/your-dataset/modele/bunny.obj")
# DRAGON_PATH     = Path("/kaggle/input/your-dataset/modele/dragon_small.obj")
# NN_MODELS_DIR   = Path("/kaggle/input/your-dataset/nn_models")

# Local:
ARMADILLO_PATH  = Path("modele/armadillo_small.obj")
TEAPOT_PATH     = Path("modele/teapot.obj")
ASIAN_DRAG_PATH = Path("modele/asian_dragon_really_small.obj")
BUNNY_PATH      = Path("modele/bunny.obj")
DRAGON_PATH     = Path("modele/dragon_small.obj")
NN_MODELS_DIR   = Path("nn_models")

print(f"Device       : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
for name, p in [
    ("Armadillo",    ARMADILLO_PATH),
    ("Teapot",       TEAPOT_PATH),
    ("Asian Dragon", ASIAN_DRAG_PATH),
    ("Bunny",        BUNNY_PATH),
    ("Dragon",       DRAGON_PATH),
    ("nn_models/",   NN_MODELS_DIR),
]:
    print(f"{name:<14}: {p}  (exists: {p.exists()})")

Device       : cpu
Armadillo     : modele\armadillo_small.obj  (exists: True)
Teapot        : modele\teapot.obj  (exists: True)
Asian Dragon  : modele\asian_dragon_really_small.obj  (exists: True)
Bunny         : modele\bunny.obj  (exists: True)
Dragon        : modele\dragon_small.obj  (exists: True)
nn_models/    : nn_models  (exists: True)


In [3]:
from armadillo.train import load_model as load_armadillo
# from bunny.train import load_model as load_bunny
# from dragon.train import load_model as load_dragon

model_armadillo = load_armadillo(
    checkpoint_path    = str(NN_MODELS_DIR / "model_armadillo.pt"),
    local_hidden_dims  = [64, 128],
    global_hidden_dims = [256, 512],
    output_hidden_dims = [256, 128],
    device             = DEVICE,
)

# model_bunny = load_bunny(
#     checkpoint_path = str(NN_MODELS_DIR / "model_bunny.pt"),
#     device          = DEVICE,
# )

# model_dragon = load_dragon(
#     checkpoint_path = str(NN_MODELS_DIR / "model_dragon.pt"),
#     device          = DEVICE,
# )

print("All models loaded.")

All models loaded.


In [4]:
from armadillo.dataset import load_obj_pointcloud, normalize_pointcloud
from armadillo.utils import visualize_transition_3d, interpolate_pointclouds

N_POINTS = 2048

def transform(obj_path: Path, model, device, n_points: int = N_POINTS) -> np.ndarray:
    """Load mesh, normalize, run through model. Returns (N, 3) predicted points."""
    pts = normalize_pointcloud(load_obj_pointcloud(str(obj_path), n_points))
    tensor = torch.from_numpy(pts).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(tensor).squeeze(0).cpu().numpy()
    return pred

def show_transition(src_path: Path, model, device, src_title: str):
    src_pts  = normalize_pointcloud(load_obj_pointcloud(str(src_path), N_POINTS))
    pred_pts = transform(src_path, model, device)
    steps    = interpolate_pointclouds(src_pts, pred_pts, steps=4)
    visualize_transition_3d(
        steps,
        titles=[src_title, "Step 1", "Step 2", "Teapot (predicted)"]
    )
    return pred_pts

teapot_pts = normalize_pointcloud(load_obj_pointcloud(str(TEAPOT_PATH), N_POINTS))

# Bunny -> Teapot

In [ ]:
print("=== Bunny → Teapot ===")
pred_bunny = show_transition(BUNNY_PATH, model_bunny, DEVICE, "Bunny (input)")

# Dragon -> Teapot

In [ ]:
print("=== Dragon → Teapot ===")
pred_dragon = show_transition(DRAGON_PATH, model_dragon, DEVICE, "Dragon (input)")

# Armadillo -> Teapot

In [5]:
print("=== Armadillo → Teapot ===")
pred_armadillo = show_transition(ARMADILLO_PATH, model_armadillo, DEVICE, "Armadillo (input)")

=== Armadillo → Teapot ===


In [8]:
print("=== Asian Dragon → Teapot (all three flows) ===")
# pred_asian_bunny     = show_transition(ASIAN_DRAG_PATH, model_bunny,     DEVICE, "Asian Dragon (bunny flow)")
# pred_asian_dragon    = show_transition(ASIAN_DRAG_PATH, model_dragon,    DEVICE, "Asian Dragon (dragon flow)")
pred_asian_armadillo = show_transition(ASIAN_DRAG_PATH, model_armadillo, DEVICE, "Asian Dragon (armadillo flow)")

=== Asian Dragon → Teapot (all three flows) ===


# Metrics

In [10]:
import sys
sys.path.insert(0, "..")
from evaluate import evaluate_all
from armadillo.utils import pointcloud_to_mesh

import pandas as pd

teapot_mesh = trimesh.load(str(TEAPOT_PATH), force="mesh")

def get_metrics(pred_pts: np.ndarray) -> dict:
    pred_mesh = pointcloud_to_mesh(pred_pts)
    return evaluate_all(pred_mesh, teapot_mesh, n_points=N_POINTS, resolution=64)

rows = [
    # {"Method": "bunny flow",                 **get_metrics(pred_bunny)},
    # {"Method": "dragon flow",                **get_metrics(pred_dragon)},
    {"Method": "armadillo flow",             **get_metrics(pred_armadillo)},
    # {"Method": "bunny flow asian dragon",    **get_metrics(pred_asian_bunny)},
    # {"Method": "dragon flow asian dragon",   **get_metrics(pred_asian_dragon)},
    {"Method": "armadillo flow asian dragon",**get_metrics(pred_asian_armadillo)},
]

df = pd.DataFrame(rows).set_index("Method")
df.columns = ["IoU", "Dice", "Chamfer"]
df = df.round({"IoU": 4, "Dice": 4, "Chamfer": 6})
print(df.to_string())

                                IoU    Dice   Chamfer
Method                                               
armadillo flow               0.7603  0.8638  3.183414
armadillo flow asian dragon  0.7473  0.8554  3.256968


In [ ]:
import numpy as np
from pathlib import Path

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# np.save(RESULTS_DIR / "pred_bunny.npy",          pred_bunny)
# np.save(RESULTS_DIR / "pred_dragon.npy",         pred_dragon)
np.save(RESULTS_DIR / "pred_armadillo.npy",      pred_armadillo)
# np.save(RESULTS_DIR / "pred_asian_bunny.npy",    pred_asian_bunny)
# np.save(RESULTS_DIR / "pred_asian_dragon.npy",   pred_asian_dragon)
np.save(RESULTS_DIR / "pred_asian_armadillo.npy", pred_asian_armadillo)